# 06 · Preferred and fallback clusters

A B300 has 148 SMs. A 2×4 cluster wants 8 CTAs at a time. 148 does not divide by 8.

`fp16_gemm_4` handles the remainder with a **fallback** cluster shape for the SMs that cannot form a preferred cluster. The obvious question — how much of your GPU ends up in the fallback path? — is invisible to kernel-level timing and trivial once each path is named.

## 1. The question

Two cluster shapes, chosen per CTA at launch:

```python
preferred_cluster_shape_mnk = (2, 4, 1)   # the ideal cluster
fallback_cluster_shape_mnk  = (2, 1, 1)   # for the leftover SMs
```

**The question:** what fraction of tile-passes run each shape, and does the fallback path cost meaningfully more?

This is a counting question, and counting is something a trace does well: label both paths and read off the counts.

### Setup

Every notebook starts here. If this machine cannot profile, the collection cells below print their command instead of failing, and the notebook still reads end to end.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "tools"))
import check_env, nbhelp, trace_summary as ts

env = check_env.report()

Environment
----------------------------------------------------------
  GPU           NVIDIA B300 SXM6 AC  [SM103 (Blackwell)]
  Driver        610.57.04
  cutlass-dsl   4.7.1
  iket API      yes
  run-iket      yes
----------------------------------------------------------
  Live profiling: AVAILABLE


## 2. The instrumentation

Each CTA knows its cluster shape by compile-time specialisation, so the two paths get different range names. `cluster_arrive` and `cluster_wait` bracket the cluster barrier; `mcast_setup` labels multicast TMA setup on the preferred path.

**The count matters more than the duration here.** A range's `count` field tells you how many times each path executed — the answer to the question — while the median tells you whether the fallback is slower.

In [2]:
nbhelp.show_iket_calls(
    nbhelp.kernel("instrumented", "fp16_gemm_4_iket_detailed.py"),
    limit=12,
)

fp16_gemm_4_iket_detailed.py: 88 IKET call sites (showing first 12)
------------------------------------------------------------------------
>  144      cute.experimental.iket.range_push("prologue")
       ...
>  180      cute.experimental.iket.range_push("mcast_setup")
       ...
>  192      cute.experimental.iket.range_pop()
       ...
>  195      cute.experimental.iket.range_push("pipeline_setup")
       ...
>  295      cute.experimental.iket.range_pop()
       ...
>  298      cute.experimental.iket.range_push("cluster_arrive")
       ...
>  300      cute.experimental.iket.range_pop()
       ...
>  386      cute.experimental.iket.range_push("cluster_wait")
       ...
>  388      cute.experimental.iket.range_pop()
       ...
>  403      cute.experimental.iket.range_pop()
       ...
>  413          cute.experimental.iket.range_push("tma_block")
       ...
>  418              cute.experimental.iket.range_push("tma_tile")


## 3. Run it

In [3]:
OUT = "out/nb06_gemm4"
cmd = (
    f"run-iket --output-dir {OUT} --clobber profile --postprocess all -- "
    f"python {nbhelp.kernel("instrumented", "fp16_gemm_4_iket_detailed.py")} --mnk 4096,4096,512"
)
check_env.run_or_show(cmd, env, est="about 1 minute")

$ run-iket --output-dir out/nb06_gemm4 --clobber profile --postprocess all -- python /path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fp16_gemm_4_iket_detailed.py --mnk 4096,4096,512

[run-iket] ============= Dry run application first to collect some info... =============


Running Blackwell fp16 GEMM example 4 (with MIX CGA support):
  mnk:                        [4096, 4096, 512]
  tolerance:                  0.1
  Preferred cluster shape:    (2, 4, 1)
  Fallback cluster shape:     (2, 1, 1)
  CTA tile shape (MNK):       (256, 256, 64)

/path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fp16_gemm_4_iket_detailed.py:841: DeprecationWarning: Migrated to examples/CuTeDSL/helpers/dynamic_persistent_tile_scheduler.py (BSD-3). The wheel copy will be removed in a future release.
  preferred_tile_sched_params = utils.ClcDynamicPersistentTileSchedulerParams(
/path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fp16_gemm_4_iket_detailed.py:859: DeprecationWarning: Migrated to examples/CuTeDSL/helpers/dynamic_persistent_tile_scheduler.py (BSD-3). The wheel copy will be removed in a future release.
  fallback_tile_sched_params = utils.ClcDynamicPersistentTileSchedulerParams(
Verifying reference result...
PASS
[run-iket] Auto-computed context buf

Running Blackwell fp16 GEMM example 4 (with MIX CGA support):
  mnk:                        [4096, 4096, 512]
  tolerance:                  0.1
  Preferred cluster shape:    (2, 4, 1)
  Fallback cluster shape:     (2, 1, 1)
  CTA tile shape (MNK):       (256, 256, 64)

/path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fp16_gemm_4_iket_detailed.py:841: DeprecationWarning: Migrated to examples/CuTeDSL/helpers/dynamic_persistent_tile_scheduler.py (BSD-3). The wheel copy will be removed in a future release.
  preferred_tile_sched_params = utils.ClcDynamicPersistentTileSchedulerParams(
/path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fp16_gemm_4_iket_detailed.py:859: DeprecationWarning: Migrated to examples/CuTeDSL/helpers/dynamic_persistent_tile_scheduler.py (BSD-3). The wheel copy will be removed in a future release.
  fallback_tile_sched_params = utils.ClcDynamicPersistentTileSchedulerParams(
Verifying reference result...
PASS


[run-iket] Dumped perfetto trace to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb06_gemm4/iket_pid_0x392036.pftrace
[run-iket] Dumped json trace to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb06_gemm4/iket_pid_0x392036.trace.json


[run-iket] Dumped compressed trace to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb06_gemm4/iket_pid_0x392036.pftrace.gz
[run-iket] Dumped perfetto HTML viewer to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb06_gemm4/iket_pid_0x392036.html
[run-iket] To view: cd /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb06_gemm4 && python3 -m http.server 8080 , then open http://localhost:8080/iket_pid_0x392036.html
[run-iket] Removing intermediate output directory: /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb06_gemm4/tracker
[run-iket] Removing intermediate output directory: /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb06_gemm4/gen-config
[run-iket] Removing intermediate output directory: /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb06_gemm4/iket


0

## 4. The trace

In [4]:
nbhelp.figure("nb06/gemm4_preferred_cluster.png", "CTAs running the preferred cluster shape.")

In [5]:
nbhelp.figure("nb06/gemm4_fallback_cluster.png", "CTAs running the fallback shape.")

In [6]:
try:
    s = ts.summarize(ts.load(ts.find_trace("out/nb06_gemm4")))
    m = {r.name: r for r in s.ranges}
    pref = m.get("preferred_cluster")
    fall = m.get("fallback_cluster")
    if pref and fall:
        tot = pref.count + fall.count
        print(f"preferred: {pref.count:5d} passes ({100*pref.count/tot:.0f}%)  median {pref.median_us:.3f} us")
        print(f"fallback : {fall.count:5d} passes ({100*fall.count/tot:.0f}%)  median {fall.median_us:.3f} us")
except FileNotFoundError:
    print("No local trace here — reference numbers below.")

preferred:   840 passes (81%)  median 13.920 us
fallback :   196 passes (19%)  median 13.776 us


## 5. The finding

Measured on a NVIDIA B300 at `mnk=4096,4096,512` — 1,036 warps, 17.60 µs span:

| Range | Count | Share | Median |
|---|---|---|---|
| `preferred_cluster` | **840** | 81% | 13.888 µs |
| `fallback_cluster` | **196** | 19% | 13.728 µs |
| `cluster_arrive` | 1,036 | — | 0.032 µs |
| `cluster_wait` | 1,036 | — | 0.032 µs |
| `mcast_setup` | 1,036 | — | 0.032 µs |


**81% preferred, 19% fallback** — a direct answer to the opening question, and one you could not get by timing the kernel.

**The fallback is not a slow path.** At 13.728 µs median it is, if anything, marginally *faster* than the preferred 13.888 µs. The fallback exists to keep leftover SMs busy, and it does that without penalty.

**The cluster barriers are free.** `cluster_arrive` and `cluster_wait` both sit at 0.032 µs — one timestamp quantum, i.e. below what this instrumentation can resolve. Whatever is costing time in this kernel, it is not cluster synchronisation.

That last point is worth dwelling on: a *negative* result, cheaply obtained, that removes a plausible suspect from the list. Not every useful profile finds a problem.

**Next:** [07 · TMA prefetch into L2](07_tma_prefetch.ipynb)

---
*Measured on a single NVIDIA B300, public `nvidia-cutlass-dsl` 4.7.1. Your absolute numbers will differ; the shapes in the trace are the transferable part.*